# 03 — Sparsity-aware splits: a default direction for missing values

*Chapter 09 · XGBoost · Notebook 3 of 5*

Real tabular data has gaps — a sensor that failed, a question left blank, a field that does not apply.
Most of the models in this course cannot fit a table with a hole in it; you have to **impute** first,
filling each gap with a guess. XGBoost does something different and, in this notebook, satisfying: at
every split it learns a **default direction** for the missing rows, and routes them there with no
imputation at all.

This is the third of XGBoost's ingredients, after the second-order view (NB 1) and the regularized
objective (NB 2). We build it by hand — trying both directions and keeping the better one — watch the
missing rows route themselves to the group they resemble, match XGBoost's learned direction exactly,
and end by checking which boosters can take a `NaN` and which cannot.

**Prerequisites.** Notebook 2 (the split gain `½[…] − γ`, and `Cover = Σ h`); Chapter 08 NB 5–6
(`HistGradientBoosting*` was named as the fast booster that accepts missing values).

**What you'll be able to do.**
- Explain how a split sends missing rows down a single learned **default direction** (Chen & Guestrin §3.4).
- Compute the split gain **both ways** and pick the direction that maximises it — by hand.
- Match XGBoost's learned default direction and its reported gain exactly.
- Say which boosters accept a `NaN` natively, and why native handling can beat impute-then-fit.

## Missing values are everywhere

Chapter 08's `GradientBoosting` cannot fit a feature matrix that contains a `NaN` — it raises an error,
and you must **impute** first: replace each gap with a column mean, a median, or a flag. Imputation is
a modelling choice, and a lossy one — it can erase the very fact that a value was missing, which is
sometimes informative (a blank income field may itself signal something).

XGBoost takes the `NaN` directly. At each split it picks not only a threshold but also a **default
direction** — the side every missing row is sent to — and it **learns** that direction from the data.
Let us build the idea on a small example.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()

SEED = 0

# A 1-feature toy whose MISSING values carry signal: the rows with no x have y ~ 3,
# so they belong with the dear group (x = 7, 8, 9), not the cheap one (x = 1, 2, 3).
x = np.array([1.0, 2.0, 3.0, 7.0, 8.0, 9.0, np.nan, np.nan]).reshape(-1, 1)
y = np.array([1.0, 1.2, 0.9, 3.0, 3.2, 2.9, 3.1, 3.0])
F0 = float(y.mean())
g = F0 - y               # gradient of 1/2 (y - F)^2 at F0
h = np.ones_like(y)      # squared error: h = 1
G, H = g.sum(), h.sum()
xv = x.ravel()
miss = np.isnan(xv)
nonmiss = ~miss

print(f"F0 = mean(y) = {F0:.4f}")
print(f"{int(miss.sum())} rows have a missing x; their y values are {y[miss]}")


## The idea: a learned default direction

At a split on feature `x`, the non-missing rows go left or right by the threshold (`x < t`). The
**missing** rows have no value to compare against — so XGBoost sends them *all* to one side, the
**default direction**, and **learns which side** by a simple rule:

> compute the split gain (Notebook 2) **twice** — once sending the missing rows left, once right — and
> keep the direction with the larger gain.

The threshold and the default direction are chosen **together**, to maximise the gain. Nothing is
imputed: the rows keep their `NaN`, and the model decides where they help most.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
low = nonmiss & (xv < 5)
high = nonmiss & (xv >= 5)
ax.scatter(xv[low], y[low], s=90, color=COLORS["class_a"], edgecolor=COLORS["text"],
           linewidth=0.5, zorder=3, label="cheap group (y ≈ 1)")
ax.scatter(xv[high], y[high], s=90, color=COLORS["class_b"], edgecolor=COLORS["text"],
           linewidth=0.5, zorder=3, label="dear group (y ≈ 3)")

# The missing rows have no x: show them in a shaded band at the right, at their y values.
x_sentinel = 11.0
ax.axvspan(10.0, 12.0, color=COLORS["muted"], alpha=0.15)
ax.scatter(np.full(int(miss.sum()), x_sentinel), y[miss], s=120, marker="s",
           color=COLORS["error"], edgecolor=COLORS["text"], linewidth=0.5, zorder=3,
           label="x missing (y ≈ 3)")
ax.text(11.0, 1.7, "x missing — which way?", ha="center", va="center", color=COLORS["text"])
ax.set_xlabel("feature x   (missing rows drawn at the right, x unknown)")
ax.set_ylabel("target y")
ax.legend(loc="center left")
plt.show()


**Read the figure.** The non-missing points fall into a cheap group (`y ≈ 1`) and a dear group
(`y ≈ 3`). The two missing rows — drawn on the right because they have no `x` — sit at `y ≈ 3`, so by
eye they belong with the dear group. But a split cannot see their `x`. Can it still send them the right
way? It can, by trying both directions and keeping the one that fits the data better.

## The recipe, by hand

For each candidate threshold (the midpoints of the sorted non-missing `x`), we compute the Notebook-2
gain **twice** — once with the missing rows added to the left child, once to the right — and keep the
`(threshold, direction)` pair with the highest gain. We reuse `λ = 1, γ = 0`, exactly as in Notebook 2,
so the gain is `½[ G_L²/(H_L+λ) + G_R²/(H_R+λ) − G²/(H+λ) ]`.

In [ ]:
def gain_for(threshold, missing_dir, lam=1.0):
    """Half-gain of the split x < threshold, sending the missing rows 'left' or 'right'."""
    left = nonmiss & (xv < threshold)
    right = nonmiss & (xv >= threshold)
    if missing_dir == "left":
        left = left | miss
    else:
        right = right | miss
    GL, HL = g[left].sum(), h[left].sum()
    GR, HR = g[right].sum(), h[right].sum()
    half_gain = 0.5 * (GL**2 / (HL + lam) + GR**2 / (HR + lam) - G**2 / (H + lam))
    return half_gain, int(left.sum()), int(right.sum())


vals = np.sort(np.unique(xv[nonmiss]))
thresholds = (vals[:-1] + vals[1:]) / 2

best = (None, None, -np.inf)
print("            missing -> left        missing -> right")
for t in thresholds:
    gl, n_l_left, n_r_left = gain_for(t, "left")
    gr, n_l_right, n_r_right = gain_for(t, "right")
    print(f"  x < {t:<4.1f}  gain {gl:6.3f} (n {n_l_left}/{n_r_left})    "
          f"gain {gr:6.3f} (n {n_l_right}/{n_r_right})")
    for direction, gain in [("left", gl), ("right", gr)]:
        if gain > best[2]:
            best = (t, direction, gain)
print()
print(f"best: x < {best[0]:.1f}, missing -> {best[1]}, half-gain = {best[2]:.4f}")


In [ ]:
gl_curve = [gain_for(t, "left")[0] for t in thresholds]
gr_curve = [gain_for(t, "right")[0] for t in thresholds]
i_best = int(np.argmax(gr_curve))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, gl_curve, "o-", color=COLORS["class_a"], linewidth=2.2, label="missing → left")
ax.plot(thresholds, gr_curve, "o-", color=COLORS["class_b"], linewidth=2.2, label="missing → right")
ax.plot(thresholds[i_best], gr_curve[i_best], "o", color=COLORS["highlight"],
        markersize=15, zorder=5,
        label=f"best: x < {thresholds[i_best]:.0f}, → right ({gr_curve[i_best]:.2f})")
ax.set_xlabel("split threshold  (x < t)")
ax.set_ylabel("half-gain")
ax.legend()
plt.show()


**Read the figure.** At every candidate threshold, sending the missing rows **right** earns more
gain than sending them left — the amber curve sits above the periwinkle one throughout. The best split
overall is `x < 5` with the missing rows going right (half-gain ≈ 2.95). The model is using the missing
rows as evidence: because their `y` matches the dear group, routing them right sharpens the split.

In [ ]:
# tree_method="exact" so the threshold search matches our by-hand midpoints (histograms: NB 4).
reg = XGBRegressor(n_estimators=1, max_depth=1, learning_rate=1.0, reg_lambda=1.0, gamma=0.0,
                   base_score=F0, objective="reg:squarederror", tree_method="exact",
                   missing=np.nan, random_state=SEED).fit(x, y)
dump = reg.get_booster().trees_to_dataframe()
root = dump[dump["Feature"] != "Leaf"].iloc[0]
default_right = root["Missing"] == root["No"]
xgb_gain = float(root["Gain"])

gl5 = gain_for(5.0, "left")[0]
gr5 = gain_for(5.0, "right")[0]

fig, ax = plt.subplots(figsize=(7.5, 5))
bars = ax.bar(["missing → left", "missing → right"], [gl5, gr5],
              color=[COLORS["class_a"], COLORS["class_b"]], width=0.6)
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.set_ylabel("half-gain at the split  x < 5")
ax.set_title(f"XGBoost learns: missing → {'right' if default_right else 'left'}")
plt.show()

cols = ["ID", "Feature", "Split", "Yes", "No", "Missing", "Gain", "Cover"]
print(dump[cols].to_string(index=False))
print()
side = "right (No)" if default_right else "left (Yes)"
print(f"XGBoost: split x < {root['Split']:.1f};  missing rows -> {root['Missing']} = {side}")
print(f"by-hand half-gain (→right) {gr5:.4f} · 2× = {2 * gr5:.4f} == XGBoost Gain {xgb_gain:.4f}")


**Read the figure.** Sending the missing rows right gives a half-gain of ≈ 2.95, far above the
≈ 1.04 of sending them left — so by hand we choose **missing → right**. XGBoost, fit on the same data
with `missing = np.nan`, learned the very same direction: its `Missing` column points to the right
child, and its reported `Gain` is `2 ×` our half-gain (the no-½ convention from Notebook 2). The two
missing rows landed in the dear-group leaf — its `Cover = 5` is the 3 dear rows plus the 2 missing
ones — with no imputation anywhere.

## Who can take a `NaN`?

Native missing-value handling is a real feature, not a given. Let us hand the same `NaN`-bearing
matrix to three boosters and see which fit and which refuse.

In [ ]:
print("Can each booster fit a matrix that contains a NaN?")
print()
candidates = [
    ("GradientBoostingRegressor", GradientBoostingRegressor(random_state=SEED)),
    ("HistGradientBoostingRegressor", HistGradientBoostingRegressor(random_state=SEED)),
    ("XGBRegressor", XGBRegressor(n_estimators=5, random_state=SEED)),
]
for name, est in candidates:
    try:
        est.fit(x, y)
        print(f"  {name:32s} ACCEPTS  (fit succeeded)")
    except ValueError as err:
        print(f"  {name:32s} REJECTS  ({str(err).splitlines()[0][:40]})")


**Read it.** Plain `GradientBoosting` **rejects** the `NaN` outright — to use it you would impute
first, and a column-mean impute discards the signal that the value was missing at all.
`HistGradientBoosting` (Chapter 08) and XGBoost **accept** it natively; XGBoost introduced the
sparsity-aware split in 2016. When missingness is informative — as in our toy, where a missing `x`
reliably means `y ≈ 3` — the learned default direction can beat impute-then-fit, and it is simpler.

*(For contrast: the classic CART answer to missing values was **surrogate splits** — a backup feature
that mimics the primary split, ESL §9.2.4. XGBoost's single learned default direction is cheaper and
scales to large data.)*

## Where this sits

Three of XGBoost's ingredients are now built by hand: the **second-order view** (NB 1), the
**regularized objective** (NB 2), and the **sparsity-aware split** (NB 3). Each is a real modelling
idea, and each you reproduced against the library.

What remains is **engineering** — chiefly the *histogram* split-finding that bins features so the
threshold search is fast at scale. That, together with the estimator's full parameter set, is the
subject of Notebook 4.

## Your turn

1. **(warm-up)** In the toy, change the two missing rows' `y` from ≈ 3 to ≈ 1, re-run the search, and
   predict the new default direction before you look. Did it flip to **left**? Why?
2. **(core)** At the threshold `x < 5`, compute `gain(missing → left)` and `gain(missing → right)` by
   hand from `G_L, H_L, G_R, H_R` (with `λ = 1`), and confirm that right wins by the margin the figure
   shows.
3. **(reach)** Explain why imputing the column mean before a `NaN`-blind model can do *worse* than
   XGBoost's native handling when the missing-ness is informative. What does the mean-impute throw away
   that the default direction keeps?

## What you built

You built XGBoost's **sparsity-aware split**: at each node, missing rows are sent down a single
**learned default direction**, chosen by computing the split gain both ways and keeping the larger.
Built by hand, it matched XGBoost exactly — the same direction (`missing → right`) and a gain equal to
`2 ×` our half-gain. And you saw the dividing line: plain `GradientBoosting` rejects a `NaN`, while
`HistGradientBoosting` and XGBoost accept it natively.

**Vocabulary.** missing / sparse value · default direction · sparsity-aware split · native `NaN`
handling · (CART) surrogate split.

Next: the estimator `XGBRegressor` / `XGBClassifier`, its full parameter set, and the **histogram**
method that makes the threshold search fast.

## References

- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD '16. DOI
  [10.1145/2939672.2939785](https://doi.org/10.1145/2939672.2939785). (§3.4, the sparsity-aware split
  and its learned default direction.)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §9.2.4 (surrogate splits — CART's classic handling of missing values). DOI
  [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).
- Friedman, J. H. (2001). *Greedy function approximation: a gradient boosting machine.* Annals of
  Statistics, 29(5), 1189–1232. DOI [10.1214/aos/1013203451](https://doi.org/10.1214/aos/1013203451).

*Previous: 02 — The regularized objective.*  ·  *Next: 04 — The estimator and its parameters.*